# 📚 Vidya — Intelligent Tutoring System with Context Pruning

## Project: The Education Tutor for Remote India

### Problem
Standard RAG systems send the same amount of context to the LLM on every query — regardless of how much of it is actually relevant. For rural India with 2G connectivity and limited compute, this is expensive and slow.

### Solution: Context Pruning
Before sending any text to the LLM, we **prune irrelevant chapters** using a lightweight classifier. Only relevant chapter text is sent — reducing tokens by 70–90% compared to baseline RAG.

### Pipeline
```
PDF → Chapter Segmentation → [Query]
                                  ↓
                         Context Pruning (cheap LLM call)
                                  ↓
                         Relevant Chapters Only
                                  ↓
                         Answer Generation (main LLM call)
                                  ↓
                         Cost Comparison vs Baseline
```

## Step 1: Install Dependencies

In [ ]:
!pip install -q langchain langchain-community langchain-groq
!pip install -q langchain-text-splitters langchain-huggingface
!pip install -q faiss-cpu pdfplumber pypdf sentence-transformers

## Step 2: Imports

In [ ]:
import os
import re
import pdfplumber
from dataclasses import dataclass, field
from typing import List, Dict, Optional

from langchain_groq import ChatGroq
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain.schema import Document

print('✅ All imports successful')

## Step 3: Configuration

In [ ]:
# ── API KEY ──────────────────────────────────────────────────────────────────
GROQ_API_KEY = "your_groq_api_key_here"  # Replace with your key
os.environ["GROQ_API_KEY"] = GROQ_API_KEY

# ── MODELS ───────────────────────────────────────────────────────────────────
PRUNER_MODEL  = "llama-3.1-8b-instant"   # Cheap model for chapter selection
ANSWER_MODEL  = "llama-3.1-8b-instant"   # Main model for answering

# ── TOKEN COST (Groq llama-3.1-8b-instant) ───────────────────────────────────
COST_PER_1K_INPUT_TOKENS  = 0.00000005   # $0.05 per 1M input tokens
COST_PER_1K_OUTPUT_TOKENS = 0.00000008   # $0.08 per 1M output tokens
CHARS_PER_TOKEN           = 4            # Approximate

print('✅ Configuration set')

## Step 4: Upload & Load the PDF

In [ ]:
from google.colab import files

print('Upload your textbook PDF...')
uploaded = files.upload()
PDF_PATH = list(uploaded.keys())[0]
print(f'✅ Uploaded: {PDF_PATH}')

## Step 5: Chapter Segmentation

Instead of splitting into arbitrary chunks (baseline RAG), we split by **chapter** — the natural unit of knowledge in a textbook.

In [ ]:
@dataclass
class Chapter:
    """Represents one chapter extracted from the textbook."""
    number:   int
    title:    str
    text:     str
    pages:    List[int] = field(default_factory=list)
    tokens:   int = 0

    def __post_init__(self):
        self.tokens = len(self.text) // 4  # approximate


def extract_chapters_from_pdf(pdf_path: str) -> List[Chapter]:
    """
    Extract text from PDF and segment into chapters.
    Detects chapter boundaries by looking for 'Chapter N' headings.
    Falls back to equal-sized page groups if no headings found.
    """
    print(f'📖 Loading PDF: {pdf_path}')
    pages_text = []

    with pdfplumber.open(pdf_path) as pdf:
        total_pages = len(pdf.pages)
        print(f'   Total pages: {total_pages}')
        for i, page in enumerate(pdf.pages):
            text = page.extract_text() or ''
            pages_text.append((i + 1, text))

    # ── Detect chapter boundaries ─────────────────────────────────────────
    chapter_pattern = re.compile(
        r'(?:^|\n)\s*(?:CHAPTER|Chapter)\s+(\d+)[:\s–-]+([^\n]{3,80})',
        re.MULTILINE
    )

    full_text = '\n'.join(t for _, t in pages_text)
    matches = list(chapter_pattern.finditer(full_text))

    chapters = []

    if len(matches) >= 2:
        print(f'   Found {len(matches)} chapter headings')
        for i, match in enumerate(matches):
            num   = int(match.group(1))
            title = match.group(2).strip()
            start = match.start()
            end   = matches[i + 1].start() if i + 1 < len(matches) else len(full_text)
            text  = full_text[start:end].strip()
            chapters.append(Chapter(number=num, title=title, text=text))
    else:
        # Fallback: split into 20-page groups
        print('   No chapter headings found — splitting into 20-page groups')
        group_size = 20
        for i in range(0, len(pages_text), group_size):
            group  = pages_text[i:i + group_size]
            text   = '\n'.join(t for _, t in group)
            pages  = [p for p, _ in group]
            num    = i // group_size + 1
            title  = f'Section {num} (pages {pages[0]}–{pages[-1]})'
            chapters.append(Chapter(number=num, title=title, text=text, pages=pages))

    total_tokens = sum(c.tokens for c in chapters)
    print(f'\n✅ Extracted {len(chapters)} chapters')
    print(f'   Total tokens (approx): {total_tokens:,}')
    print()
    for ch in chapters:
        print(f'   Ch.{ch.number:2d} | {ch.title[:55]:55s} | ~{ch.tokens:,} tokens')
    return chapters


chapters = extract_chapters_from_pdf(PDF_PATH)

## Step 6: Build Baseline RAG (for comparison)

This is exactly what your original notebook did — chunking + vector search. We keep it to measure cost.

In [ ]:
print('Building baseline RAG vector store...')

# Load all docs the standard way
loader   = PyPDFLoader(PDF_PATH)
all_docs = loader.load()

# Split into fixed-size chunks (standard baseline approach)
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)
chunks = splitter.split_documents(all_docs)
print(f'   Total chunks: {len(chunks)}')

# Embed and store
embeddings   = HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')
vectorstore  = FAISS.from_documents(chunks, embeddings)

BASELINE_K        = 5                                    # chunks retrieved per query
BASELINE_TOKENS   = BASELINE_K * (1000 // 4)            # fixed cost every query

print(f'\n✅ Baseline RAG ready')
print(f'   Fixed cost per query: ~{BASELINE_TOKENS:,} tokens (always, regardless of question)')

## Step 7: Context Pruning Engine

This is the key technique. Instead of searching vectors, we ask a **cheap LLM** to look at chapter titles and decide which ones are relevant. Only those chapters are included in the final prompt.

In [ ]:
# Initialise LLM clients
pruner_llm = ChatGroq(
    api_key=GROQ_API_KEY,
    model_name=PRUNER_MODEL,
    temperature=0,
    max_tokens=100      # Only needs to return a short JSON list
)

answer_llm = ChatGroq(
    api_key=GROQ_API_KEY,
    model_name=ANSWER_MODEL,
    temperature=0.2,
    max_tokens=512
)


def prune_chapters(question: str, chapters: List[Chapter]) -> List[int]:
    """
    CONTEXT PRUNING: Use a cheap LLM call (only chapter titles, no full text)
    to identify which chapters are relevant to the question.

    Cost: tiny — we only send chapter titles, not their content.
    Returns: list of relevant chapter indices.
    """
    chapter_list = '\n'.join(
        f'{i}: Chapter {ch.number} — "{ch.title}"'
        for i, ch in enumerate(chapters)
    )

    pruning_prompt = f"""You are a context-pruning engine for an educational AI tutor.
Given a student question and a list of textbook chapter titles, identify ONLY the
chapter numbers (by index) that are DIRECTLY relevant to answering the question.

Return ONLY a Python list of integer indices. No explanation. No other text.
Example: [0, 3]

Student question: "{question}"

Chapter list:
{chapter_list}

Relevant indices:"""

    response = pruner_llm.invoke(pruning_prompt)
    text     = response.content.strip()

    # Parse the list
    try:
        match   = re.search(r'\[([\d,\s]+)\]', text)
        indices = [int(x.strip()) for x in match.group(1).split(',')]
        indices = [i for i in indices if 0 <= i < len(chapters)]  # validate
        return indices if indices else [0]
    except Exception:
        # Keyword fallback if LLM returns unexpected format
        q_lower = question.lower()
        return [
            i for i, ch in enumerate(chapters)
            if any(word in q_lower for word in ch.title.lower().split())
        ][:2] or [0]


print('✅ Context Pruning engine ready')

## Step 8: Answer Generation with Pruned Context

In [ ]:
def answer_with_pruning(question: str, chapters: List[Chapter]) -> Dict:
    """
    Full Context Pruning pipeline:
    1. Prune: identify relevant chapters (cheap call, titles only)
    2. Answer: send only relevant chapter text to LLM
    3. Return answer + cost metrics
    """
    # ── Stage 1: Context Pruning ─────────────────────────────────────────
    relevant_indices = prune_chapters(question, chapters)
    pruned_indices   = [i for i in range(len(chapters)) if i not in relevant_indices]

    relevant_chapters = [chapters[i] for i in relevant_indices]
    pruned_chapters   = [chapters[i] for i in pruned_indices]

    # ── Stage 2: Build pruned context ────────────────────────────────────
    context = '\n\n'.join(
        f'=== Chapter {ch.number}: {ch.title} ===\n{ch.text[:3000]}'
        for ch in relevant_chapters
    )

    answer_prompt = f"""You are Vidya, a friendly AI tutor helping Class 10 students in rural India.
Answer the question using ONLY the textbook content provided below.
Be clear, accurate, and concise. Use simple language.

Textbook content:
{context}

Student question: {question}

Answer:"""

    response = answer_llm.invoke(answer_prompt)
    answer   = response.content.strip()

    # ── Stage 3: Calculate token costs ──────────────────────────────────
    pruned_tokens   = len(context)     // CHARS_PER_TOKEN
    baseline_tokens = sum(ch.tokens for ch in chapters)   # if we had sent everything
    tokens_saved    = baseline_tokens - pruned_tokens
    savings_pct     = round(tokens_saved / baseline_tokens * 100, 1) if baseline_tokens else 0

    pruned_cost     = pruned_tokens   * COST_PER_1K_INPUT_TOKENS
    baseline_cost   = baseline_tokens * COST_PER_1K_INPUT_TOKENS

    return {
        'answer':            answer,
        'relevant_chapters': relevant_chapters,
        'pruned_chapters':   pruned_chapters,
        'pruned_tokens':     pruned_tokens,
        'baseline_tokens':   baseline_tokens,
        'tokens_saved':      tokens_saved,
        'savings_pct':       savings_pct,
        'pruned_cost':       pruned_cost,
        'baseline_cost':     baseline_cost,
    }


def answer_baseline(question: str) -> Dict:
    """
    Standard RAG baseline (your original code) — for cost comparison.
    """
    docs    = vectorstore.similarity_search(question, k=BASELINE_K)
    context = '\n'.join(d.page_content for d in docs)

    prompt  = f"""You are an AI tutor. Use the context to answer the question.\n\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"""
    response = answer_llm.invoke(prompt)

    tokens = len(context) // CHARS_PER_TOKEN
    return {
        'answer': response.content.strip(),
        'tokens': tokens,
        'cost':   tokens * COST_PER_1K_INPUT_TOKENS
    }


print('✅ Answer functions ready')

## Step 9: Test the System — Side-by-Side Comparison

In [ ]:
def compare(question: str):
    """Run both systems and print a side-by-side cost comparison."""
    print('=' * 70)
    print(f'QUESTION: {question}')
    print('=' * 70)

    # ── Baseline RAG ────────────────────────────────────────────────────
    print('\n[BASELINE RAG]')
    base = answer_baseline(question)
    print(f'Tokens used  : {base["tokens"]:,}')
    print(f'Est. cost    : ${base["cost"]:.6f}')
    print(f'Answer       : {base["answer"][:300]}...')

    # ── Context Pruning ──────────────────────────────────────────────────
    print('\n[CONTEXT PRUNING]')
    result = answer_with_pruning(question, chapters)

    kept    = ', '.join(f'Ch.{c.number}' for c in result['relevant_chapters'])
    skipped = ', '.join(f'Ch.{c.number}' for c in result['pruned_chapters'])

    print(f'Kept chapters: {kept}')
    print(f'Pruned       : {skipped}')
    print(f'Tokens used  : {result["pruned_tokens"]:,}')
    print(f'Baseline was : {result["baseline_tokens"]:,}')
    print(f'Tokens saved : {result["tokens_saved"]:,}  ({result["savings_pct"]}% reduction)')
    print(f'Est. cost    : ${result["pruned_cost"]:.6f}  (was ${result["baseline_cost"]:.6f})')
    print(f'Answer       : {result["answer"][:300]}...')
    print()

    return result


# ── Run test questions ────────────────────────────────────────────────────────
test_questions = [
    'What are micelles and how do soaps work?',
    'What is photosynthesis?',
    'What is plaster of paris and how is it made?',
    'Explain Ohm\'s law with an example.',
    'How does the human eye work?',
]

results = []
for q in test_questions:
    r = compare(q)
    results.append(r)

## Step 10: Cost Summary Report

In [ ]:
print('=' * 70)
print('  COST SUMMARY REPORT — Context Pruning vs Baseline RAG')
print('=' * 70)
print(f'{"Question":<45} {"Baseline":>10} {"Pruned":>10} {"Saved":>8}')
print('-' * 70)

total_baseline = 0
total_pruned   = 0

for q, r in zip(test_questions, results):
    b = r['baseline_tokens']
    p = r['pruned_tokens']
    s = r['savings_pct']
    total_baseline += b
    total_pruned   += p
    print(f'{q[:44]:<45} {b:>10,} {p:>10,} {s:>7.1f}%')

overall_saving = round((total_baseline - total_pruned) / total_baseline * 100, 1)
print('-' * 70)
print(f'{"TOTAL":<45} {total_baseline:>10,} {total_pruned:>10,} {overall_saving:>7.1f}%')
print()
print(f'  Total tokens saved  : {total_baseline - total_pruned:,}')
print(f'  Overall reduction   : {overall_saving}%')
print(f'  Baseline total cost : ${total_baseline * COST_PER_1K_INPUT_TOKENS:.6f}')
print(f'  Pruned total cost   : ${total_pruned   * COST_PER_1K_INPUT_TOKENS:.6f}')
print(f'  Money saved         : ${(total_baseline - total_pruned) * COST_PER_1K_INPUT_TOKENS:.6f}')
print('=' * 70)

## Step 11: Interactive Tutor Loop

In [ ]:
print('🌿 Vidya — Interactive Tutor with Context Pruning')
print('   Type your question and press Enter. Type "exit" to stop.\n')

session_tokens_baseline = 0
session_tokens_pruned   = 0
query_count             = 0

while True:
    question = input('\nAsk a question: ').strip()
    if question.lower() in ('exit', 'quit', 'q'):
        break
    if not question:
        continue

    query_count += 1
    result = answer_with_pruning(question, chapters)

    session_tokens_baseline += result['baseline_tokens']
    session_tokens_pruned   += result['pruned_tokens']

    kept = ', '.join(f'Ch.{c.number} ({c.title})' for c in result['relevant_chapters'])

    print(f'\n✂️  Pruned to: {kept}')
    print(f'   Tokens used: {result["pruned_tokens"]:,}  '
          f'(saved {result["savings_pct"]}% vs baseline)')
    print(f'\n📖 Answer:')
    print(result['answer'])
    print(f'\n── Session total: {query_count} queries | '
          f'{session_tokens_pruned:,} tokens used | '
          f'{round((session_tokens_baseline-session_tokens_pruned)/session_tokens_baseline*100,1)}% saved overall')